# FunnyBirds CBM — Counterfactual Concept Swap Analysis

Three-level causal analysis of species-identity leakage in the standard CBM:

| Level | Method | Null |
|---|---|---|
| **Causal** | Concept swap: replace `z[c]` in species-A with values from species-B | `leakage_sym = 0` |
| **Mechanistic** | Species-identity probe: linear `z → species_id` | accuracy = 1/50 = 0.02 |
| **Discriminative** | Binary probe per pair: `z_c scalar → {A, B}` | accuracy = 0.5 |

**Requirements**
- `checkpoints_funnybirds/cbm_funnybirds.pth`
- `features/resnet50_cbm_funnybirds/avgpool_{train,test}.pt`
- `features/resnet50_cbm_funnybirds/labels_{train,test}.pt`
- `data/FunnyBirds/metadata/` (from `prepare_funnybirds_metadata.py`)

In [ ]:
import random
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 40)
print('imports OK')

## Config

Edit `ROOT` to match your setup.  All other paths are derived from it.

In [ ]:
ROOT      = Path('/scratch/network/cr7998/cv_emergence_project')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FB        = ROOT / 'data'     / 'FunnyBirds'
CBM_FEATS = ROOT / 'features' / 'resnet50_cbm_funnybirds'
CBM_CKPT  = ROOT / 'checkpoints_funnybirds' / 'cbm_funnybirds.pth'

N_SPECIES  = 50
N_CONCEPTS = 26

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
for label, p in [('FB', FB), ('CBM_FEATS', CBM_FEATS), ('CBM_CKPT', CBM_CKPT)]:
    print(f'  {label}: exists={p.exists()}  ({p})')

## §1  Metadata & class-concept matrix

FunnyBirds has an exact 50×26 class-concept matrix: `cc_matrix[s, c] = 1` iff
species `s` truly has concept `c`.  Every row sums to 5 (one variant per part).
This lets us enumerate GT-positive pairs precisely — no annotation noise.

Metadata CSVs are loaded from `FB/metadata/` (generated by `prepare_funnybirds_metadata.py`).

In [ ]:
def load_species_maps(fb_root: Path):
    classes_csv = fb_root / 'metadata' / 'classes.csv'
    if not classes_csv.exists():
        raise FileNotFoundError(f'metadata/classes.csv not found under {fb_root}')
    df = pd.read_csv(classes_csv)
    id2name  = dict(zip(df['class_id'], df['class_name']))
    id2short = {k: v.replace('funnybird_', 'FB') for k, v in id2name.items()}
    return id2name, id2short


def load_meta(fb_root: Path) -> pd.DataFrame:
    images_csv = fb_root / 'metadata' / 'images.csv'
    if not images_csv.exists():
        raise FileNotFoundError(f'metadata/images.csv not found under {fb_root}')
    df = pd.read_csv(images_csv)
    id2name, _ = load_species_maps(fb_root)
    df['species_id']   = df['class_id']
    df['species_name'] = df['class_id'].map(id2name)
    return df


def load_image_attr_labels_robust(fb_root: Path) -> pd.DataFrame:
    concepts_csv = fb_root / 'metadata' / 'image_concepts_binary.csv'
    if not concepts_csv.exists():
        raise FileNotFoundError(
            f'metadata/image_concepts_binary.csv not found under {fb_root}. '
            'Run prepare_funnybirds_metadata.py first.'
        )
    wide = pd.read_csv(concepts_csv)
    concept_cols = [c for c in wide.columns if c != 'image_id']
    long = wide.melt(id_vars='image_id', value_vars=concept_cols,
                     var_name='attr_name', value_name='is_present')
    long['attr_id']    = long.groupby('attr_name', sort=False).ngroup()
    long['is_present'] = long['is_present'].astype(int)
    long['certainty']  = 1   # FunnyBirds: ground-truth, always certain
    return long[['image_id', 'attr_id', 'attr_name', 'is_present', 'certainty']]


def load_attr_maps(fb_root: Path):
    concepts_csv = fb_root / 'metadata' / 'concepts.csv'
    if not concepts_csv.exists():
        raise FileNotFoundError(
            f'metadata/concepts.csv not found under {fb_root}. '
            'Run prepare_funnybirds_metadata.py first.'
        )
    df = pd.read_csv(concepts_csv)
    id2name  = dict(zip(df['concept_id'], df['concept_name']))
    name2id  = dict(zip(df['concept_name'], df['concept_id']))
    return id2name, name2id


print('Defined: load_species_maps  load_meta  load_image_attr_labels_robust  load_attr_maps')

In [ ]:
from datasets.funnybirds_dataset import concept_names as _fb_concept_names
ATTR_LIST = _fb_concept_names()
print(f'FunnyBirds concepts ({len(ATTR_LIST)}):')
for a in ATTR_LIST:
    print(f'  {a}')

In [ ]:
from datasets.funnybirds_dataset import FunnyBirdsDataset

_fb_ds    = FunnyBirdsDataset(FB, split='train')
cc_raw, _ = _fb_ds.get_class_concept_matrix()   # [50, 26]  values in {0, 1}
cc_matrix = cc_raw.float()                       # torch.Tensor [50, 26]

cc_df = pd.DataFrame(
    cc_matrix.numpy(),
    columns=ATTR_LIST,
    index=[f'funnybird_{i:02d}' for i in range(cc_matrix.shape[0])],
)
print(f'Class-concept matrix: {cc_df.shape}')
print('Row sums (each species has exactly 5 concepts):')
print(cc_df.sum(axis=1).value_counts().to_dict())
cc_df.head()

In [ ]:
meta          = load_meta(FB)
img_attr_long = load_image_attr_labels_robust(FB)
attr_id_to_name, attr_name_to_id = load_attr_maps(FB)

id2name, _  = load_species_maps(FB)
_sid_min    = int(meta['species_id'].min())   # offset for indexing into label_head (may be 0 or 1)

def spname(sid: int) -> str:
    return id2name.get(int(sid), f'funnybird_{int(sid):02d}')

print(f'meta: {len(meta)} images  '
      f'(train={int(meta["is_train"].sum())}  test={int((meta["is_train"]==0).sum())})')
print(f'concepts loaded: {len(attr_name_to_id)}')
print(f'species_id range: {_sid_min} → {int(meta["species_id"].max())}  (_sid_min={_sid_min})')

## §2  Feature loading & CBM weights

`labels_{split}.pt` stores the DataLoader image-ID ordering so features can be
aligned to metadata rows by `image_id`.

The CBM weight matrices extracted from the checkpoint:

| Tensor | Shape | Role |
|--------|-------|------|
| `W_c`  | [26, 2048] | `concept_head` linear weight |
| `b_c`  | [26]       | `concept_head` bias |
| `W_y`  | [50, 26]   | `label_head` linear weight |
| `b_y`  | [50]       | `label_head` bias |

`z = σ(avgpool @ W_c.T + b_c)` ∈ [0,1]²⁶ is the concept bottleneck.

In [ ]:
def safe_torch_load(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def to_1d_int_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    return np.array(x).reshape(-1).astype(int)


def infer_kind(arr):
    if arr.max() <= 200 and arr.min() >= 0:
        return 'species_id_like'
    if arr.max() > 200:
        return 'image_id_like'
    return 'unknown'


def load_split_order(feat_dir: Path, split: str):
    p = feat_dir / f'labels_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    t = safe_torch_load(p)
    assert isinstance(t, dict), f'Expected dict in {p}, got {type(t)}'
    assert 'image_ids' in t, f'{p} missing image_ids key; has {list(t.keys())}'
    ids  = to_1d_int_array(t['image_ids'])
    kind = infer_kind(ids)
    return kind, ids


def load_features(feat_dir: Path, layer: str, split: str) -> torch.Tensor:
    p = feat_dir / f'{layer}_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    X = safe_torch_load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()


print('Defined: safe_torch_load  to_1d_int_array  infer_kind  load_split_order  load_features')

In [ ]:
ckpt = safe_torch_load(CBM_CKPT)
sd   = ckpt['model_state_dict']

W_c = sd['concept_head.weight'].float()   # [26, 2048]
b_c = sd['concept_head.bias'].float()     # [26]
W_y = sd['label_head.weight'].float()     # [50, 26]
b_y = sd['label_head.bias'].float()       # [50]

print(f'W_c: {tuple(W_c.shape)}  b_c: {tuple(b_c.shape)}')
print(f'W_y: {tuple(W_y.shape)}  b_y: {tuple(b_y.shape)}')
if 'config' in ckpt:
    print(f"config: {ckpt['config']}")

In [ ]:
# Split order — must be image_id_like so we can align to metadata
kind_tr, ids_tr = load_split_order(CBM_FEATS, 'train')
kind_te, ids_te = load_split_order(CBM_FEATS, 'test')
assert kind_tr == 'image_id_like', f'Expected image_id_like, got {kind_tr}'
assert kind_te == 'image_id_like', f'Expected image_id_like, got {kind_te}'

# Raw backbone features
X_tr = load_features(CBM_FEATS, 'avgpool', 'train')   # [N_tr, 2048]
X_te = load_features(CBM_FEATS, 'avgpool', 'test')    # [N_te, 2048]
print(f'avgpool train: {tuple(X_tr.shape)}')
print(f'avgpool test:  {tuple(X_te.shape)}')

# Concept activations — exact forward pass of CBMFunnyBirds
with torch.no_grad():
    z_tr = torch.sigmoid(X_tr @ W_c.T + b_c)    # [N_tr, 26]
    z_te = torch.sigmoid(X_te @ W_c.T + b_c)    # [N_te, 26]
    logits_tr = z_tr @ W_y.T + b_y              # [N_tr, 50]
    logits_te = z_te @ W_y.T + b_y              # [N_te, 50]

print(f'z_tr: {tuple(z_tr.shape)}  range [{z_tr.min():.3f}, {z_tr.max():.3f}]')
print(f'z_te: {tuple(z_te.shape)}  range [{z_te.min():.3f}, {z_te.max():.3f}]')

# Species labels aligned to split order
meta_idx   = meta.set_index('image_id')
species_tr = np.array([int(meta_idx.loc[int(i), 'species_id']) for i in ids_tr], dtype=int)
species_te = np.array([int(meta_idx.loc[int(i), 'species_id']) for i in ids_te], dtype=int)

# 0-based class indices (match the label_head output dimension 0..N_SPECIES-1)
species_tr_0 = species_tr - _sid_min
species_te_0 = species_te - _sid_min

print(f'species_tr_0 range: {species_tr_0.min()} → {species_tr_0.max()}')
print(f'species_te_0 range: {species_te_0.min()} → {species_te_0.max()}')

In [ ]:
# ── Species accuracy ──────────────────────────────────────────────────────────
with torch.no_grad():
    pred_te = logits_te.argmax(dim=1).numpy()
species_acc = float((pred_te == species_te_0).mean())
print(f'CBM species accuracy (test): {species_acc:.4f}  (chance=1/{N_SPECIES}={1/N_SPECIES:.4f})')

# ── Concept accuracy ──────────────────────────────────────────────────────────
# GT concept label for each test image: the row of cc_matrix for its species
gt_c_te = np.stack([cc_matrix[s].numpy() for s in species_te_0])   # [N_te, 26]
z_te_bin = (z_te.numpy() > 0.5).astype(float)
concept_acc = float((z_te_bin == gt_c_te).mean())
print(f'Concept binary accuracy (test): {concept_acc:.4f}  (chance=0.5)')

per_c_acc = (z_te_bin == gt_c_te).mean(axis=0)
print('\nPer-concept accuracy:')
for name, acc in zip(ATTR_LIST, per_c_acc):
    print(f'  {name:<20s}  {acc:.4f}')

## §3  Counterfactual concept swap

For a pair (A, B) both GT-positive for concept `c`:

**Forward swap** (B donates to A):  
For each A-image, replace `z[c]` with the value from each B-image in turn, then average over B-images.

```
delta_B_fwd = mean_A[ P(class=B | z_A with z_A[c]←z_B[c]) − P(class=B | z_A) ]
delta_A_fwd = mean_A[ P(class=A | z_A with z_A[c]←z_B[c]) − P(class=A | z_A) ]
leakage_fwd = delta_B_fwd − delta_A_fwd
```

**Backward swap** (A donates to B):  
Same in the other direction.  
`leakage_bwd = delta_A_bwd − delta_B_bwd`

**Symmetric leakage** (main metric):  
`leakage_sym = 0.5 × (leakage_fwd + leakage_bwd)`

- `leakage_sym > 0` → swapping concept `c` shifts the model's prediction toward the donor species
- `leakage_sym ≈ 0` → concept `c` does not carry species-identity signal

**Class index note:** `sid_0 = species_id − _sid_min` indexes the label-head's 50-class softmax.

In [ ]:
@torch.no_grad()
def concept_swap(
    z_donor: torch.Tensor,   # [N_D, K]  source of replacement values for concept c
    z_recip: torch.Tensor,   # [N_R, K]  images whose concept c is replaced
    concept_idx: int,
    W_y: torch.Tensor,       # [C, K]
    b_y: torch.Tensor,       # [C]
):
    """
    Replace z_recip[:, concept_idx] with each z_donor row's value in turn,
    compute softmax, and average over donors.

    Returns
    -------
    p_orig : [N_R, C]   softmax of original z_recip
    p_swap : [N_R, C]   softmax after swap, averaged over N_D donors
    """
    N_R, K = z_recip.shape
    N_D    = z_donor.shape[0]
    C      = b_y.shape[0]

    p_orig = torch.softmax(z_recip @ W_y.T + b_y, dim=-1)           # [N_R, C]

    # Expand recipients: N_R copies, one per donor
    z_sw   = z_recip.unsqueeze(1).expand(N_R, N_D, K).clone()       # [N_R, N_D, K]
    z_sw[:, :, concept_idx] = (
        z_donor[:, concept_idx]                                       # [N_D]
        .unsqueeze(0).expand(N_R, N_D)                               # [N_R, N_D]
    )
    logits_sw = z_sw.reshape(N_R * N_D, K) @ W_y.T + b_y            # [N_R*N_D, C]
    p_swap = (
        torch.softmax(logits_sw, dim=-1)
        .reshape(N_R, N_D, C)
        .mean(dim=1)                                                  # [N_R, C]
    )
    return p_orig, p_swap


print('Defined: concept_swap')

In [ ]:
def pair_swap_metrics(
    z_A: torch.Tensor,   # [N_A, K]  concept activations for all A test-images
    z_B: torch.Tensor,   # [N_B, K]  concept activations for all B test-images
    concept_idx: int,
    W_y: torch.Tensor,
    b_y: torch.Tensor,
    sid_A_0: int,        # 0-based class index (= species_id - _sid_min)
    sid_B_0: int,
    concept_name: str,
) -> dict:
    """
    Symmetric leakage for a GT-positive pair (A, B) on concept concept_idx.

    Forward (B donates to A):
        delta_B_fwd  mean change in P(class=B) for A-images after swap
        delta_A_fwd  mean change in P(class=A) for A-images after swap
        leakage_fwd = delta_B_fwd - delta_A_fwd

    Backward (A donates to B):
        delta_A_bwd  mean change in P(class=A) for B-images after swap
        delta_B_bwd  mean change in P(class=B) for B-images after swap
        leakage_bwd = delta_A_bwd - delta_B_bwd

    leakage_sym = 0.5 * (leakage_fwd + leakage_bwd)
    """
    # Forward: B donates concept c to A-images
    p_A_orig, p_A_swap = concept_swap(z_B, z_A, concept_idx, W_y, b_y)
    delta_B_fwd = float((p_A_swap[:, sid_B_0] - p_A_orig[:, sid_B_0]).mean())
    delta_A_fwd = float((p_A_swap[:, sid_A_0] - p_A_orig[:, sid_A_0]).mean())
    leakage_fwd = delta_B_fwd - delta_A_fwd

    # Backward: A donates concept c to B-images
    p_B_orig, p_B_swap = concept_swap(z_A, z_B, concept_idx, W_y, b_y)
    delta_A_bwd = float((p_B_swap[:, sid_A_0] - p_B_orig[:, sid_A_0]).mean())
    delta_B_bwd = float((p_B_swap[:, sid_B_0] - p_B_orig[:, sid_B_0]).mean())
    leakage_bwd = delta_A_bwd - delta_B_bwd

    leakage_sym = 0.5 * (leakage_fwd + leakage_bwd)

    return {
        'concept':     concept_name,
        'concept_idx': int(concept_idx),
        'sid_A':       int(sid_A_0 + _sid_min),   # original species_id (for display)
        'sid_B':       int(sid_B_0 + _sid_min),
        'species_A':   spname(sid_A_0 + _sid_min),
        'species_B':   spname(sid_B_0 + _sid_min),
        'n_A':         int(z_A.shape[0]),
        'n_B':         int(z_B.shape[0]),
        'delta_B_fwd': float(delta_B_fwd),
        'delta_A_fwd': float(delta_A_fwd),
        'leakage_fwd': float(leakage_fwd),
        'delta_A_bwd': float(delta_A_bwd),
        'delta_B_bwd': float(delta_B_bwd),
        'leakage_bwd': float(leakage_bwd),
        'leakage_sym': float(leakage_sym),
    }


print('Defined: pair_swap_metrics')

## §4  Run all GT-positive pairs

For each of the 26 concepts, `cc_matrix` gives all GT-positive species.
We enumerate every pair and run the symmetric swap.

FunnyBirds has 10 test images per species, so each species contributes 10 recipient images
and 10 donor images.  The swap averages over all 10 donors, so each recipient gets a single
averaged prediction shift.

In [ ]:
# GT-positive species per concept (0-based class indices)
concept_gt_pos = {}
for c_idx, c_name in enumerate(ATTR_LIST):
    pos_sids_0 = [s for s in range(N_SPECIES) if float(cc_matrix[s, c_idx]) > 0.5]
    concept_gt_pos[c_name] = pos_sids_0

for c_name, sids in concept_gt_pos.items():
    n_pairs = len(sids) * (len(sids) - 1) // 2
    print(f'{c_name:<20s}  {len(sids):2d} GT-pos species  {n_pairs:4d} pairs')

total_pairs = sum(len(v)*(len(v)-1)//2 for v in concept_gt_pos.values())
print(f'\nTotal pairs across all concepts: {total_pairs}')

In [ ]:
all_rows = []

for c_idx, c_name in enumerate(ATTR_LIST):
    pos_sids_0 = concept_gt_pos[c_name]
    n_pairs    = len(pos_sids_0) * (len(pos_sids_0) - 1) // 2
    print(f'[{c_idx:2d}] {c_name:<20s}  {len(pos_sids_0)} pos species  {n_pairs} pairs', end='  ')

    for sid_A_0, sid_B_0 in combinations(pos_sids_0, 2):
        z_A = z_te[species_te_0 == sid_A_0]   # [n_A, 26]
        z_B = z_te[species_te_0 == sid_B_0]   # [n_B, 26]
        if z_A.shape[0] == 0 or z_B.shape[0] == 0:
            print(f'[warn] empty: sid_A_0={sid_A_0} sid_B_0={sid_B_0}')
            continue
        row = pair_swap_metrics(z_A, z_B, c_idx, W_y, b_y,
                                sid_A_0=sid_A_0, sid_B_0=sid_B_0,
                                concept_name=c_name)
        all_rows.append(row)
    print('done')

swap_df = pd.DataFrame(all_rows)
swap_df.to_csv('fb_cbm_counterfactual_swap.csv', index=False)
print(f'\nTotal rows: {len(swap_df)}  — saved fb_cbm_counterfactual_swap.csv')
swap_df[['concept','species_A','species_B','leakage_fwd','leakage_bwd','leakage_sym']].head(8)

## §5  Aggregation and visualisation

`leakage_sym` is averaged over all GT-positive pairs for each concept.
A high mean for `beak_0` means `z[beak_0]` encodes *which* beak variant the bird has,
not just *whether* it has that variant.  That is concept backwash.

In [ ]:
PART_ORDER  = ['beak', 'wing', 'tail', 'feet', 'eye']
PART_COLORS = {'beak':'#E15759','wing':'#4E79A7','tail':'#F28E2B',
               'feet':'#76B7B2','eye':'#59A14F','other':'#BAB0AC'}

def concept_part(c: str) -> str:
    for p in PART_ORDER:
        if c.startswith(p):
            return p
    return 'other'

swap_df['part'] = swap_df['concept'].apply(concept_part)

# Per-concept aggregation in canonical ATTR_LIST order
concept_agg = (
    swap_df.groupby('concept', as_index=False)
    .agg(
        leakage_sym_mean  = ('leakage_sym', 'mean'),
        leakage_sym_median= ('leakage_sym', 'median'),
        leakage_sym_std   = ('leakage_sym', 'std'),
        leakage_fwd_mean  = ('leakage_fwd', 'mean'),
        leakage_bwd_mean  = ('leakage_bwd', 'mean'),
        n_pairs           = ('leakage_sym', 'size'),
    )
)
concept_agg['part'] = concept_agg['concept'].apply(concept_part)
# Preserve canonical order
concept_agg = concept_agg.set_index('concept').loc[ATTR_LIST].reset_index()

bar_colors = [PART_COLORS[p] for p in concept_agg['part']]

fig, ax = plt.subplots(figsize=(14, 4))
xs = np.arange(len(concept_agg))
ax.bar(xs, concept_agg['leakage_sym_mean'],
       color=bar_colors, yerr=concept_agg['leakage_sym_std'],
       capsize=3, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(xs)
ax.set_xticklabels(concept_agg['concept'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean leakage_sym  (± std over GT-positive pairs)')
ax.set_title('FunnyBirds CBM — Counterfactual leakage per concept\n'
             '(positive = swapping concept activations shifts predictions toward the donor species)')
ax.legend(handles=[Patch(color=PART_COLORS[p], label=p) for p in PART_ORDER],
          title='Body part', fontsize=8, ncol=5, loc='upper right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fb_cbm_leakage_per_concept.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_leakage_per_concept.png')
display(concept_agg[['concept','part','n_pairs','leakage_sym_mean','leakage_sym_std']])

In [ ]:
# Decompose leakage_sym into its two directions
fig, ax = plt.subplots(figsize=(6, 6))
for part, grp in swap_df.groupby('part'):
    ax.scatter(grp['leakage_fwd'], grp['leakage_bwd'],
               alpha=0.35, s=14, label=part, color=PART_COLORS.get(part, 'gray'))

all_vals = pd.concat([swap_df['leakage_fwd'], swap_df['leakage_bwd']])
lo, hi   = float(all_vals.min()) - 0.005, float(all_vals.max()) + 0.005
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8, label='diagonal (symmetric)')
ax.axhline(0, color='gray', linewidth=0.5, linestyle=':')
ax.axvline(0, color='gray', linewidth=0.5, linestyle=':')
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel('leakage_fwd  (B donates to A)')
ax.set_ylabel('leakage_bwd  (A donates to B)')
ax.set_title('Leakage asymmetry\n(on-diagonal = symmetric; off-diagonal = directional)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('fb_cbm_leakage_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_leakage_decomposition.png')

In [ ]:
part_agg = (
    swap_df.groupby('part', as_index=False)
    .agg(
        leakage_sym_mean = ('leakage_sym', 'mean'),
        leakage_sym_sem  = ('leakage_sym', lambda x: x.std() / max(np.sqrt(len(x)), 1)),
        n_pairs          = ('leakage_sym', 'size'),
    )
    .set_index('part').reindex(PART_ORDER).reset_index()
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(part_agg['part'], part_agg['leakage_sym_mean'],
       yerr=part_agg['leakage_sym_sem'], capsize=4,
       color=[PART_COLORS[p] for p in part_agg['part']],
       alpha=0.85, edgecolor='black')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Mean leakage_sym  (± SEM)')
ax.set_title('Leakage by body part — FunnyBirds CBM')
ax.grid(True, axis='y', alpha=0.3)
for i, row in part_agg.iterrows():
    ax.text(i, float(row['leakage_sym_mean']) + float(row['leakage_sym_sem']) + 2e-4,
            f"n={int(row['n_pairs'])}", ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('fb_cbm_leakage_by_part.png', dpi=150, bbox_inches='tight')
plt.show()
display(part_agg)

## §5d  Per-pair leakage distribution & per-species ranking

Are there specific species that systematically inflate or suppress leakage?
- **Left**: full histogram of `leakage_sym` over all (A, B, concept) triples.
  If backwash is species-specific we expect a right tail — not all-negative.
- **Right**: mean `leakage_sym` per species (averaged over every concept and pair they appear in),
  ranked highest → lowest.  Red = net positive leaker.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full histogram of per-pair leakage_sym
ax   = axes[0]
vals = swap_df['leakage_sym'].values
frac_pos = float((vals > 0).mean())
ax.hist(vals, bins=60, color='steelblue', alpha=0.75, edgecolor='white')
ax.axvline(0,           color='black',  lw=1.5, ls='--', label='0')
ax.axvline(vals.mean(), color='crimson', lw=1.5, ls='-',
           label=f'mean = {vals.mean():.4f}')
ax.text(0.97, 0.93,
        f'frac > 0: {frac_pos:.1%}\n({int((vals>0).sum())} / {len(vals)} pairs)',
        transform=ax.transAxes, fontsize=9, ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
ax.set_xlabel('leakage_sym per (A, B, concept) triple')
ax.set_ylabel('Count')
ax.set_title('Per-pair leakage distribution\n'
             'If backwash is species-specific: expect right tail, not all-negative')
ax.legend(); ax.grid(True, alpha=0.3)

# Right: per-species mean leakage, ranked
ax = axes[1]
sp_leak = (
    pd.concat([
        swap_df[['sid_A','leakage_sym']].rename(columns={'sid_A':'sid'}),
        swap_df[['sid_B','leakage_sym']].rename(columns={'sid_B':'sid'}),
    ])
    .groupby('sid')['leakage_sym'].mean()
    .reset_index(name='mean_leakage')
    .sort_values('mean_leakage', ascending=False)
    .reset_index(drop=True)
)
sp_colors = ['crimson' if v > 0 else 'steelblue' for v in sp_leak['mean_leakage']]
ax.scatter(range(len(sp_leak)), sp_leak['mean_leakage'], c=sp_colors, s=50, zorder=3)
ax.axhline(0, color='black', lw=1, ls='--')
for rank, row in sp_leak.head(5).iterrows():
    ax.annotate(f"sp{int(row['sid'])}", (rank, row['mean_leakage']),
                textcoords='offset points', xytext=(3, 4), fontsize=7)
for rank, row in sp_leak.tail(5).iterrows():
    ax.annotate(f"sp{int(row['sid'])}", (rank, row['mean_leakage']),
                textcoords='offset points', xytext=(3, -10), fontsize=7)
ax.set_xlabel('Species rank (highest → lowest mean leakage_sym)')
ax.set_ylabel('Mean leakage_sym')
ax.set_title('Per-species mean leakage (averaged over all concepts & pairs)\n'
             'red = net positive leaker')
ax.grid(True, alpha=0.3)

plt.suptitle('FunnyBirds CBM: Is leakage heterogeneous across species?', y=1.02)
plt.tight_layout()
plt.savefig('fb_cbm_leakage_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_leakage_dist.png')

# Expose ranking for spotlight cells below
print('\nTop 10 species by mean leakage_sym:')
display(sp_leak.head(10).assign(species=sp_leak.head(10)['sid'].apply(spname)))

## §5e  Spotlight: net-positive leaker species

Drill into the species with **positive mean `leakage_sym`** (sp29, sp39 from the scatter above).

For each spotlight species we show:
1. GT concept profile + mean `z` value across test images
2. Per-concept breakdown of `leakage_sym` — which body-part concept drives the positive leakage?
3. Top (concept, partner) triples — which pairs cause the largest positive shift?
4. All 10 test images (to visually inspect the bird phenotype)

In [ ]:
# Determine spotlight species from sp_leak (top positive leakers)
SPOTLIGHT_SIDS = sp_leak[sp_leak['mean_leakage'] > 0]['sid'].tolist()
if not SPOTLIGHT_SIDS:
    # fallback: top-2 by mean_leakage even if negative
    SPOTLIGHT_SIDS = sp_leak.head(2)['sid'].tolist()
print(f'Spotlight species (positive mean leakage_sym): {SPOTLIGHT_SIDS}')

# ── 1. Concept profiles + mean z ─────────────────────────────────────────────
print('\nGT concept profiles and mean test-set z values:')
for sid in SPOTLIGHT_SIDS:
    sid_0  = int(sid) - _sid_min
    pos_c  = [ATTR_LIST[i] for i in range(N_CONCEPTS) if float(cc_matrix[sid_0, i]) > 0.5]
    mask   = (species_te_0 == sid_0)
    z_mean = z_te[mask].mean(dim=0).numpy()   # mean z over this species' test images
    print(f'\n  sp{sid}  ({spname(int(sid))}):')
    print(f'  {"concept":<20s}  {"mean z_te":>9s}  GT')
    for c in ATTR_LIST:
        ci  = ATTR_LIST.index(c)
        gt  = int(float(cc_matrix[sid_0, ci]) > 0.5)
        marker = '◀ GT=1' if gt else ''
        print(f'  {c:<20s}  {z_mean[ci]:>9.3f}  {marker}')

# ── 2. Per-concept leakage breakdown ─────────────────────────────────────────
fig, axes = plt.subplots(1, len(SPOTLIGHT_SIDS), figsize=(7*len(SPOTLIGHT_SIDS), 4.5),
                          squeeze=False)
for col, sid in enumerate(SPOTLIGHT_SIDS):
    mask_A = swap_df['sid_A'] == sid
    mask_B = swap_df['sid_B'] == sid
    sp_rows = pd.concat([
        swap_df[mask_A][['concept','leakage_sym','species_B']].rename(columns={'species_B':'partner'}),
        swap_df[mask_B][['concept','leakage_sym','species_A']].rename(columns={'species_A':'partner'}),
    ], ignore_index=True)

    by_concept = (sp_rows.groupby('concept')['leakage_sym']
                  .mean().reindex(ATTR_LIST).fillna(0))
    c_colors   = [PART_COLORS.get(concept_part(c), 'gray') for c in by_concept.index]

    ax = axes[0, col]
    ax.bar(range(len(by_concept)), by_concept.values, color=c_colors, alpha=0.85, edgecolor='white')
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_xticks(range(len(by_concept)))
    ax.set_xticklabels(by_concept.index, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Mean leakage_sym')
    ax.set_title(f'sp{sid}  ({spname(int(sid))})')
    ax.legend(handles=[Patch(color=PART_COLORS[p], label=p) for p in PART_ORDER],
              fontsize=7, ncol=5, loc='best')
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Per-concept leakage for net-positive leakers', y=1.02)
plt.tight_layout()
plt.savefig('fb_cbm_spotlight_concepts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_spotlight_concepts.png')

# ── 3. Top (concept, partner) triples ────────────────────────────────────────
for sid in SPOTLIGHT_SIDS:
    mask_A = swap_df['sid_A'] == sid
    mask_B = swap_df['sid_B'] == sid
    sp_rows = pd.concat([
        swap_df[mask_A][['concept','leakage_sym','leakage_fwd','leakage_bwd','species_B']]
                .rename(columns={'species_B':'partner'}),
        swap_df[mask_B][['concept','leakage_sym','leakage_fwd','leakage_bwd','species_A']]
                .rename(columns={'species_A':'partner'}),
    ], ignore_index=True)
    print(f'\n── sp{sid} ({spname(int(sid))}): top 10 (concept, partner) by leakage_sym ──')
    display(sp_rows.nlargest(10, 'leakage_sym')
            [['concept','partner','leakage_sym','leakage_fwd','leakage_bwd']]
            .reset_index(drop=True))

# ── 4. Sample test images ─────────────────────────────────────────────────────
try:
    from torchvision import transforms as T_vis
    tf_vis  = T_vis.Compose([T_vis.Resize((128, 128)), T_vis.ToTensor()])
    vis_ds  = FunnyBirdsDataset(FB, split='test', transform=tf_vis, include_concepts=False)

    cls_to_idx = {}
    for i in range(len(vis_ds)):
        lbl = int(vis_ds[i]['label'])
        cls_to_idx.setdefault(lbl, []).append(i)

    N_COLS = 10
    fig, axes = plt.subplots(len(SPOTLIGHT_SIDS), N_COLS,
                              figsize=(N_COLS * 1.6, len(SPOTLIGHT_SIDS) * 2))
    if len(SPOTLIGHT_SIDS) == 1:
        axes = axes[np.newaxis, :]   # keep 2-D indexing

    for row_i, sid in enumerate(SPOTLIGHT_SIDS):
        sid_0 = int(sid) - _sid_min
        idxs  = cls_to_idx.get(sid_0, [])[:N_COLS]
        for col_i in range(N_COLS):
            ax = axes[row_i, col_i]
            if col_i < len(idxs):
                img = vis_ds[idxs[col_i]]['image'].permute(1, 2, 0).numpy()
                ax.imshow(np.clip(img, 0, 1))
            ax.axis('off')
        axes[row_i, 0].set_ylabel(f'sp{sid}', fontsize=11, labelpad=4)

    plt.suptitle(
        'All test images — ' + '  &  '.join(f'sp{s} ({spname(int(s))})' for s in SPOTLIGHT_SIDS),
        y=1.01,
    )
    plt.tight_layout()
    plt.savefig('fb_cbm_spotlight_images.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_cbm_spotlight_images.png')
except Exception as e:
    print(f'Image display skipped: {e}')
    print(f'FunnyBirds test images expected under: {FB}')

## §6  Species-identity probe on z

Three linear probes `→ species_id` (50-class):

| Probe | Input dim | Interpretation |
|---|---|---|
| Full z | 26 | How much species identity survives the bottleneck? |
| Per-concept | 1 (scalar z_c) | Which concept carries the most species info? |
| Avgpool | 2048 | Upper bound — how much is in the backbone? |

If **full-z accuracy ≈ avgpool accuracy**, the 26-dim bottleneck retains nearly
all species information.  That is the strongest form of the backwash hypothesis.

In [ ]:
def train_linear_probe_multiclass(
    Xtr, ytr, Xte, yte, *,
    epochs=15, lr=3e-3, wd=1e-4, seed=0,
) -> float:
    """
    Multiclass linear probe; returns test accuracy.
    Verbatim from fb_recallv2.py.
    """
    torch.manual_seed(seed)
    _dev  = 'cuda' if torch.cuda.is_available() else 'cpu'
    Xtr_t = torch.as_tensor(Xtr, dtype=torch.float32, device=_dev)
    ytr_t = torch.as_tensor(ytr, dtype=torch.long,    device=_dev)
    Xte_t = torch.as_tensor(Xte, dtype=torch.float32, device=_dev)
    d, C  = Xtr_t.shape[1], int(ytr_t.max().item()) + 1
    model = nn.Linear(d, C).to(_dev)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xtr_t), ytr_t).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(Xte_t).argmax(dim=1).cpu().numpy()
    return float((pred == np.asarray(yte)).mean())


print('Defined: train_linear_probe_multiclass')

In [ ]:
ytr_sp = species_tr_0   # 0-based species labels for train split
yte_sp = species_te_0   # 0-based species labels for test  split

# 1. Full z probe: [N, 26] → 50 classes
full_z_acc = train_linear_probe_multiclass(
    z_tr.numpy(), ytr_sp,
    z_te.numpy(), yte_sp,
    epochs=15,
)
print(f'Full z  (26-dim) → species accuracy: {full_z_acc:.4f}  (chance={1/N_SPECIES:.4f})')

# 2. Per-concept scalar probe: z[:, c] → 50 classes
per_concept_sp_acc = {}
for c_idx, c_name in enumerate(ATTR_LIST):
    acc = train_linear_probe_multiclass(
        z_tr[:, c_idx:c_idx+1].numpy(), ytr_sp,
        z_te[:, c_idx:c_idx+1].numpy(), yte_sp,
        epochs=15,
    )
    per_concept_sp_acc[c_name] = acc
    print(f'  {c_name:<20s}  {acc:.4f}')

In [ ]:
# 3. Avgpool baseline: [N, 2048] → 50 classes
avgpool_acc = train_linear_probe_multiclass(
    X_tr.numpy(), ytr_sp,
    X_te.numpy(), yte_sp,
    epochs=15,
)
print(f'Avgpool (2048-dim) → species accuracy: {avgpool_acc:.4f}')
print(f'Full z  (26-dim)   → species accuracy: {full_z_acc:.4f}')
print(f'Chance             = {1/N_SPECIES:.4f}')
print(f'Retention ratio (full_z / avgpool): {full_z_acc / max(avgpool_acc, 1e-6):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Panel A: Full z vs avgpool vs chance
ax = axes[0]
probe_labels = ['avgpool\n(2048-dim)', 'full z\n(26-dim)', 'chance\n(1/50)']
probe_accs   = [avgpool_acc, full_z_acc, 1/N_SPECIES]
probe_colors = ['#4E79A7', '#E15759', 'lightgray']
bars = ax.bar(probe_labels, probe_accs, color=probe_colors, edgecolor='black', width=0.5)
for bar, acc in zip(bars, probe_accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=11)
ax.axhline(1/N_SPECIES, color='gray', linestyle='--', linewidth=0.8, label='chance (1/50)')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Species classification accuracy')
ax.set_title('How much species identity is in z?\n(full z ≈ avgpool = concept backwash)')
ax.grid(True, axis='y', alpha=0.3)

# Panel B: Per-concept species probe accuracy
ax = axes[1]
c_names  = list(per_concept_sp_acc.keys())
c_accs   = [per_concept_sp_acc[c] for c in c_names]
c_colors = [PART_COLORS.get(concept_part(c), 'gray') for c in c_names]
ax.bar(c_names, c_accs, color=c_colors, edgecolor='white', alpha=0.85)
ax.axhline(1/N_SPECIES, color='gray', linestyle='--', linewidth=0.8, label='chance (1/50)')
ax.set_xticks(range(len(c_names)))
ax.set_xticklabels(c_names, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Species accuracy  (z_c scalar → 50 classes)')
ax.set_title('Which concept scalar carries most species identity?')
legend_handles = [Patch(color=PART_COLORS[p], label=p) for p in PART_ORDER]
legend_handles.append(Line2D([0],[0], color='gray', linestyle='--', label='chance'))
ax.legend(handles=legend_handles, fontsize=7, ncol=3)
ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Species-identity probe on CBM bottleneck z  (FunnyBirds)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fb_cbm_species_probe.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_species_probe.png')

## §7  Binary per-pair discriminability in z_c

For each GT-positive pair (A, B) and concept `c`:
train a logistic regression on the scalar `z[c]` to classify `{A, B}`.

- **Train split**: training-set images from species A and B
- **Test split**: test-set images from species A and B
- **Metric**: balanced accuracy (average of TPR and TNR, to handle equal class sizes)
- **Random baseline**: 0.5

Accuracy >> 0.5 means the single activation `z[c]` separates the two species mechanistically.

In [ ]:
def balanced_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=int).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=int).reshape(-1)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return 0.5 * (tpr + tnr)


def binary_disc_for_pair(
    z_c_tr: np.ndarray, y_tr: np.ndarray,
    z_c_te: np.ndarray, y_te: np.ndarray,
    *, epochs: int = 15, lr: float = 1e-2, wd: float = 1e-4, seed: int = 0,
) -> float:
    """Logistic regression on a scalar z_c → {0,1}; returns balanced accuracy."""
    torch.manual_seed(seed)
    _dev  = 'cuda' if torch.cuda.is_available() else 'cpu'
    Xtr_t = torch.as_tensor(z_c_tr.reshape(-1, 1), dtype=torch.float32, device=_dev)
    ytr_t = torch.as_tensor(y_tr,                  dtype=torch.float32, device=_dev).view(-1, 1)
    Xte_t = torch.as_tensor(z_c_te.reshape(-1, 1), dtype=torch.float32, device=_dev)
    model = nn.Linear(1, 1).to(_dev)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xtr_t), ytr_t).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t)).view(-1).cpu().numpy()
    pred = (probs >= 0.5).astype(int)
    return balanced_accuracy(np.asarray(y_te, dtype=int), pred)


disc_rows = []
for c_idx, c_name in enumerate(ATTR_LIST):
    for sid_A_0, sid_B_0 in combinations(concept_gt_pos[c_name], 2):
        # Train split
        m_A_tr = (species_tr_0 == sid_A_0)
        m_B_tr = (species_tr_0 == sid_B_0)
        z_c_A_tr = z_tr[m_A_tr, c_idx].numpy()
        z_c_B_tr = z_tr[m_B_tr, c_idx].numpy()
        if len(z_c_A_tr) == 0 or len(z_c_B_tr) == 0:
            continue

        # Test split
        m_A_te = (species_te_0 == sid_A_0)
        m_B_te = (species_te_0 == sid_B_0)
        z_c_A_te = z_te[m_A_te, c_idx].numpy()
        z_c_B_te = z_te[m_B_te, c_idx].numpy()
        if len(z_c_A_te) == 0 or len(z_c_B_te) == 0:
            continue

        z_c_tr_pair = np.concatenate([z_c_A_tr, z_c_B_tr])
        z_c_te_pair = np.concatenate([z_c_A_te, z_c_B_te])
        y_tr_pair   = np.concatenate([np.zeros(len(z_c_A_tr)), np.ones(len(z_c_B_tr))])
        y_te_pair   = np.concatenate([np.zeros(len(z_c_A_te)), np.ones(len(z_c_B_te))])

        disc_acc = binary_disc_for_pair(z_c_tr_pair, y_tr_pair, z_c_te_pair, y_te_pair)

        disc_rows.append({
            'concept':    c_name,
            'concept_idx':c_idx,
            'sid_A':      int(sid_A_0 + _sid_min),
            'sid_B':      int(sid_B_0 + _sid_min),
            'species_A':  spname(sid_A_0 + _sid_min),
            'species_B':  spname(sid_B_0 + _sid_min),
            'disc_acc':   float(disc_acc),
        })

disc_df = pd.DataFrame(disc_rows)
print(f'Binary discriminability computed for {len(disc_df)} (concept, pair) combinations')
disc_df[['concept','species_A','species_B','disc_acc']].head(10)

In [ ]:
# Merge leakage and discriminability on canonical pair key
# Normalise pair key so sid_lo <= sid_hi regardless of enumeration order
def norm_pair_key(df):
    df = df.copy()
    df['sid_lo'] = df[['sid_A','sid_B']].min(axis=1)
    df['sid_hi'] = df[['sid_A','sid_B']].max(axis=1)
    return df

merged = (
    norm_pair_key(swap_df)
    .merge(norm_pair_key(disc_df)[['concept','sid_lo','sid_hi','disc_acc']],
           on=['concept','sid_lo','sid_hi'], how='inner')
)
merged['part'] = merged['concept'].apply(concept_part)
print(f'Merged rows: {len(merged)}')

fig, ax = plt.subplots(figsize=(7, 5))
for part, grp in merged.groupby('part'):
    ax.scatter(grp['disc_acc'], grp['leakage_sym'],
               alpha=0.35, s=14, label=part, color=PART_COLORS.get(part, 'gray'))

if len(merged) > 5:
    xs = merged['disc_acc'].values
    ys = merged['leakage_sym'].values
    m, b_ = np.polyfit(xs, ys, 1)
    xline = np.linspace(xs.min(), xs.max(), 100)
    ax.plot(xline, m*xline + b_, 'k--', linewidth=1.5, label=f'trend (slope={m:.4f})')

ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.axvline(0.5, color='gray', linestyle=':', linewidth=0.8, label='binary chance (0.5)')
ax.set_xlabel('Binary discriminability  (z_c → {A,B}, balanced acc)')
ax.set_ylabel('leakage_sym  (causal swap metric)')
ax.set_title('Mechanistic discriminability vs causal leakage\n'
             '(strong correlation = consistent multi-level evidence)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fb_cbm_leakage_vs_discriminability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_leakage_vs_discriminability.png')

## §8  Three-level evidence per concept

Aggregate across pairs to get one row per concept, then correlate all three signals.

If `funnybirds_cbm_recall.ipynb` has been run, load the species recall CSV to add
the behavioural recall-range as the third axis.

In [ ]:
# Per-concept aggregation (mean over all GT-positive pairs)
concept_leakage = (
    merged.groupby('concept', as_index=False)
    .agg(
        leakage_sym_mean = ('leakage_sym', 'mean'),
        disc_acc_mean    = ('disc_acc',    'mean'),
        n_pairs          = ('leakage_sym', 'size'),
    )
)
concept_leakage['part'] = concept_leakage['concept'].apply(concept_part)

# Behavioural: recall range from funnybirds_cbm_recall.ipynb
RECALL_CSV   = Path('fb_cbm_species.csv')
recall_range = None
if RECALL_CSV.exists():
    cbm_sp = pd.read_csv(RECALL_CSV)
    if 'attr' in cbm_sp.columns and 'recall' in cbm_sp.columns:
        rr = (
            cbm_sp.groupby('attr')['recall']
            .agg(recall_range=lambda x: float(x.max() - x.min()))
            .reset_index()
            .rename(columns={'attr': 'concept'})
        )
        concept_leakage = concept_leakage.merge(rr, on='concept', how='left')
        recall_range = concept_leakage['recall_range']
        print(f'Loaded recall ranges from {RECALL_CSV}')
    else:
        print(f'[warn] {RECALL_CSV} missing expected columns (attr, recall)')
else:
    print(f'[warn] {RECALL_CSV} not found — run funnybirds_cbm_recall.ipynb first.')
    concept_leakage['recall_range'] = np.nan

display(concept_leakage[['concept','part','n_pairs','leakage_sym_mean','disc_acc_mean','recall_range']])

In [ ]:
has_recall = concept_leakage['recall_range'].notna().any()
ncols      = 3 if has_recall else 2
fig, axes  = plt.subplots(1, ncols, figsize=(6*ncols, 5))

def annotate_concept(ax, df, x_col, y_col):
    for _, row in df.iterrows():
        ax.annotate(
            row['concept'].split('_')[1],
            (row[x_col], row[y_col]),
            fontsize=6, ha='left', xytext=(3, 3), textcoords='offset points',
        )

def add_trend(ax, xs, ys):
    if len(xs) > 3:
        m, b_ = np.polyfit(xs, ys, 1)
        xline  = np.linspace(xs.min(), xs.max(), 100)
        ax.plot(xline, m*xline + b_, 'k--', linewidth=1.3, label=f'trend (slope={m:.4f})')

# Panel 1: causal vs mechanistic
ax = axes[0]
for part, grp in concept_leakage.groupby('part'):
    ax.scatter(grp['disc_acc_mean'], grp['leakage_sym_mean'],
               s=60, alpha=0.85, label=part, color=PART_COLORS.get(part, 'gray'))
annotate_concept(ax, concept_leakage, 'disc_acc_mean', 'leakage_sym_mean')
add_trend(ax, concept_leakage['disc_acc_mean'].values, concept_leakage['leakage_sym_mean'].values)
ax.axvline(0.5, color='gray', linestyle=':', linewidth=0.8)
ax.axhline(0,   color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Mean disc_acc  (mechanistic)')
ax.set_ylabel('Mean leakage_sym  (causal)')
ax.set_title('Causal vs mechanistic\n(per concept)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 2: causal vs species probe per concept
ax = axes[1]
sp_probe_arr = np.array([per_concept_sp_acc.get(c, np.nan) for c in concept_leakage['concept']])
for part, grp in concept_leakage.groupby('part'):
    idx  = concept_leakage['part'] == part
    ax.scatter(sp_probe_arr[idx], concept_leakage.loc[idx, 'leakage_sym_mean'],
               s=60, alpha=0.85, label=part, color=PART_COLORS.get(part, 'gray'))
for i, row in concept_leakage.iterrows():
    ax.annotate(row['concept'].split('_')[1], (sp_probe_arr[i], row['leakage_sym_mean']),
                fontsize=6, ha='left', xytext=(3, 3), textcoords='offset points')
valid = ~np.isnan(sp_probe_arr)
add_trend(ax, sp_probe_arr[valid], concept_leakage.loc[valid, 'leakage_sym_mean'].values)
ax.axvline(1/N_SPECIES, color='gray', linestyle=':', linewidth=0.8, label='species chance')
ax.axhline(0,           color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Per-concept species probe acc  (mechanistic II)')
ax.set_ylabel('Mean leakage_sym  (causal)')
ax.set_title('Causal vs species-probe accuracy\n(per concept)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 3: causal vs behavioural (if available)
if has_recall and ncols == 3:
    ax = axes[2]
    sub = concept_leakage.dropna(subset=['recall_range'])
    for part, grp in sub.groupby('part'):
        ax.scatter(grp['leakage_sym_mean'], grp['recall_range'],
                   s=60, alpha=0.85, label=part, color=PART_COLORS.get(part, 'gray'))
    annotate_concept(ax, sub, 'leakage_sym_mean', 'recall_range')
    add_trend(ax, sub['leakage_sym_mean'].values, sub['recall_range'].values)
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)
    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax.set_xlabel('Mean leakage_sym  (causal)')
    ax.set_ylabel('Recall range  (behavioural)')
    ax.set_title('Causal vs behavioural\n(per concept)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('FunnyBirds CBM — Three-level evidence for concept-identity leakage',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fb_cbm_three_level_evidence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_three_level_evidence.png')

## §9  Summary

### Metric reference

| Metric | Null | Direction | Where computed |
|---|---|---|---|
| `leakage_sym` | 0 | > 0 = backwash | §4 swap |
| `disc_acc` (binary) | 0.5 | > 0.5 = leakage | §7 |
| Per-concept species probe | 1/50 = 0.02 | >> 0.02 = leakage | §6 |
| Full-z species probe | 1/50 = 0.02 | → avgpool = full backwash | §6 |
| Recall range | 0 | > 0 = leakage | external |

### Interpretation

- **`leakage_sym >> 0`** — Swapping just that one concept activation visibly
  shifts the species prediction. Concept `c` is not just a beak-shape detector;
  it knows *which species* has that beak shape.

- **High `disc_acc` + high `leakage_sym`** — Linear and causal evidence agree.
  The activation `z[c]` is not a binary concept indicator; it is a species-specific
  real number.

- **Full-z probe ≈ avgpool probe** — The 26-dim bottleneck retains almost as much
  species identity as the 2048-dim backbone features.  The concept layer is acting
  as a compressed species embedding, not a concept detector.

### Extension to MCBM

To test whether the IB penalty (γ) reduces leakage, run the same analysis on:
- Features: `features/resnet50_funnybirds_mcbm_gamma{g}/avgpool_{split}.pt`
- Checkpoint: weights extracted from the MCBM checkpoint for gamma `g`

A decreasing `leakage_sym` vs γ is the main prediction of the IB-reduces-backwash hypothesis.